# Triplet Text Similarity and CDP Visualization

This notebook builds Visual Genome-style subject-predicate-object text prompts from `filter_total.csv`, encodes them with the repository CLIP text encoder, compares original cosine similarity with Class Diversity Promotion (CDP/SVD) refined features, and visualizes the triplet text space with PCA and optional t-SNE.

If CLIP weights are not already cached locally, `clip.load(...)` may need network access once to download the model weights.

In [ ]:
from pathlib import Path

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "maskrcnn_benchmark").exists():
    REPO_ROOT = REPO_ROOT.parent

RELATION_CSV_PATH = REPO_ROOT / "maskrcnn_benchmark/modeling/roi_heads/relation_head/filter_total.csv"
PROMPT_TEMPLATE = "a photo of a {subject} {predicate} a {object}"

# Use None to encode every generated valid triplet. Keep a cap for fast iteration.
MAX_TRIPLETS_PER_PREDICATE = 80
HEATMAP_MAX_PER_PREDICATE = 8
RANDOM_SEED = 7

# CDP removes the first k principal directions from the text feature matrix.
CDP_REMOVE_COMPONENTS = 1
CENTER_BEFORE_CDP = False

CLIP_MODEL_NAME = "ViT-B/32"
CLIP_BATCH_SIZE = 256
USE_TSNE = True
TSNE_PERPLEXITY = 30

OUTPUT_DIR = REPO_ROOT / "visualization/outputs/triplet_text_similarity"
EMBEDDING_CACHE_PATH = OUTPUT_DIR / "triplet_clip_text_embeddings.npz"

# `all_objects` matches the plan: subjects come from the CSV object vocabulary.
# `allowed_objects` is useful for faster, more conservative exploratory plots.
SUBJECT_SOURCE = "all_objects"  # one of: all_objects, allowed_objects

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", REPO_ROOT)
print("relation csv:", RELATION_CSV_PATH)
print("output dir:", OUTPUT_DIR)

In [ ]:
import hashlib
import json
import math
import random
import sys
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    sns = None

try:
    import plotly.express as px
except Exception:
    px = None

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
})

## Build Triplet Prompts

`filter_total.csv` is treated as a predicate-object validity table. Each non-background cell indicates that the row predicate is valid for that object column. For each predicate, this notebook pairs valid objects with subjects from the configured subject vocabulary and then stratified-samples per predicate if `MAX_TRIPLETS_PER_PREDICATE` is set.

In [ ]:
def clean_name(value):
    return str(value).strip()


def relation_name_from_row(row):
    values = [clean_name(v) for v in row.values]
    values = [v for v in values if v and v != "__background__" and v.lower() != "nan"]
    if not values:
        return None
    return Counter(values).most_common(1)[0][0]


def build_triplet_records(csv_path):
    df = pd.read_csv(csv_path, index_col=0)
    object_names = [clean_name(c) for c in df.columns if clean_name(c) != "__background__"]
    records = []
    predicate_counts = []
    rng = random.Random(RANDOM_SEED)

    for _, row in df.iterrows():
        predicate = relation_name_from_row(row)
        if predicate is None or predicate == "__background__":
            continue

        valid_objects = []
        for col, value in zip(df.columns, row.values):
            col_name = clean_name(col)
            if col_name == "__background__":
                continue
            if clean_name(value) == predicate:
                valid_objects.append(col_name)

        if not valid_objects:
            continue

        if SUBJECT_SOURCE == "all_objects":
            subject_candidates = object_names
        elif SUBJECT_SOURCE == "allowed_objects":
            subject_candidates = valid_objects
        else:
            raise ValueError("Unsupported SUBJECT_SOURCE: " + str(SUBJECT_SOURCE))

        all_pairs = [(subject, obj) for subject in subject_candidates for obj in valid_objects if subject != "__background__" and obj != "__background__"]
        total_count = len(all_pairs)
        if MAX_TRIPLETS_PER_PREDICATE is not None and total_count > MAX_TRIPLETS_PER_PREDICATE:
            all_pairs = rng.sample(all_pairs, int(MAX_TRIPLETS_PER_PREDICATE))
        all_pairs = sorted(all_pairs)

        for subject, obj in all_pairs:
            prompt = PROMPT_TEMPLATE.format(subject=subject, predicate=predicate, object=obj)
            records.append({
                "subject": subject,
                "predicate": predicate,
                "object": obj,
                "text": prompt,
            })
        predicate_counts.append({
            "predicate": predicate,
            "valid_objects": len(valid_objects),
            "candidate_triplets": total_count,
            "sampled_triplets": len(all_pairs),
        })

    records_df = pd.DataFrame(records)
    counts_df = pd.DataFrame(predicate_counts).sort_values("predicate").reset_index(drop=True)
    return df, object_names, records_df, counts_df


relation_df, object_names, triplets_df, predicate_counts_df = build_triplet_records(RELATION_CSV_PATH)

print("predicate rows:", len(predicate_counts_df))
print("object classes:", len(object_names))
print("constructed triplets:", len(triplets_df))
print("candidate triplets before sampling:", int(predicate_counts_df["candidate_triplets"].sum()))
display(predicate_counts_df.head(12))
display(triplets_df.head(12))

## Encode Text with CLIP

The embeddings are cached in `EMBEDDING_CACHE_PATH`. If the current text list differs from the cache, the notebook re-encodes and overwrites the cache.

In [ ]:
def text_fingerprint(texts):
    payload = "\n".join(texts).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def l2_normalize(x, eps=1e-12):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, eps)


texts = triplets_df["text"].tolist()
predicates = triplets_df["predicate"].tolist()
fingerprint = text_fingerprint(texts)

embeddings = None
if EMBEDDING_CACHE_PATH.exists():
    cache = np.load(EMBEDDING_CACHE_PATH, allow_pickle=False)
    cached_fingerprint = str(cache["fingerprint"].item()) if "fingerprint" in cache.files else ""
    if cached_fingerprint == fingerprint:
        embeddings = cache["embeddings"].astype(np.float32)
        print("loaded cached embeddings:", EMBEDDING_CACHE_PATH)
    else:
        print("cache exists but text fingerprint changed; re-encoding")

if embeddings is None:
    import torch
    sys.path.insert(0, str(REPO_ROOT))
    from CLIP import clip

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("loading CLIP", CLIP_MODEL_NAME, "on", device)
    print("If this fails because weights are missing, allow CLIP to download weights once or pre-populate the cache.")
    model, _ = clip.load(CLIP_MODEL_NAME, device=device)
    model.eval()

    encoded_batches = []
    with torch.no_grad():
        for start in range(0, len(texts), CLIP_BATCH_SIZE):
            batch_texts = texts[start:start + CLIP_BATCH_SIZE]
            tokens = clip.tokenize(batch_texts, truncate=True).to(device)
            batch_features = model.encode_text(tokens).float()
            batch_features = batch_features / batch_features.norm(dim=-1, keepdim=True).clamp(min=1e-6)
            encoded_batches.append(batch_features.cpu().numpy().astype(np.float32))
            print(f"encoded {min(start + CLIP_BATCH_SIZE, len(texts))}/{len(texts)}")

    embeddings = np.concatenate(encoded_batches, axis=0)
    np.savez_compressed(
        EMBEDDING_CACHE_PATH,
        embeddings=embeddings.astype(np.float32),
        texts=np.array(texts),
        predicates=np.array(predicates),
        fingerprint=np.array(fingerprint),
        clip_model=np.array(CLIP_MODEL_NAME),
    )
    print("saved cache:", EMBEDDING_CACHE_PATH)

embeddings = l2_normalize(embeddings.astype(np.float32))
print("embedding shape:", embeddings.shape)

## CDP/SVD Refinement

Following Class Diversity Promotion, remove the top singular vector direction from the text feature span and normalize again.

In [ ]:
def remove_top_svd_components(x, remove_components=1, center=False):
    if remove_components <= 0:
        return l2_normalize(x.copy()), np.empty((0, x.shape[1]), dtype=x.dtype), None
    base = x.astype(np.float64)
    mean = base.mean(axis=0, keepdims=True) if center else np.zeros((1, base.shape[1]), dtype=base.dtype)
    work = base - mean
    _, singular_values, vt = np.linalg.svd(work, full_matrices=False)
    components = vt[:remove_components]
    projection = work @ components.T @ components
    refined = work - projection
    if center:
        refined = refined + mean
    return l2_normalize(refined.astype(np.float32)), components.astype(np.float32), singular_values


def cosine_similarity_matrix(x):
    x = l2_normalize(x)
    return x @ x.T


def off_diagonal_values(sim):
    mask = ~np.eye(sim.shape[0], dtype=bool)
    return sim[mask]


def similarity_summary(name, sim):
    values = off_diagonal_values(sim)
    return {
        "name": name,
        "mean": float(values.mean()),
        "std": float(values.std()),
        "min": float(values.min()),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p50": float(np.percentile(values, 50)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(values.max()),
    }


refined_embeddings, removed_components, singular_values = remove_top_svd_components(
    embeddings,
    remove_components=CDP_REMOVE_COMPONENTS,
    center=CENTER_BEFORE_CDP,
)

original_sim = cosine_similarity_matrix(embeddings)
refined_sim = cosine_similarity_matrix(refined_embeddings)
summary_df = pd.DataFrame([
    similarity_summary("Original CLIP text", original_sim),
    similarity_summary(f"CDP/SVD remove top {CDP_REMOVE_COMPONENTS}", refined_sim),
])
display(summary_df)

if singular_values is not None:
    explained = singular_values ** 2 / np.maximum(np.sum(singular_values ** 2), 1e-12)
    print("top singular explained ratios:", np.round(explained[:10], 6))
    print("mean similarity drop:", summary_df.loc[0, "mean"] - summary_df.loc[1, "mean"])

## Similarity Heatmaps

The heatmap uses a smaller stratified sample per predicate so the figure remains readable.

In [ ]:
def stratified_indices(labels, max_per_label, seed=RANDOM_SEED):
    rng = random.Random(seed)
    grouped = defaultdict(list)
    for idx, label in enumerate(labels):
        grouped[label].append(idx)
    selected = []
    for label in sorted(grouped):
        indices = grouped[label]
        if max_per_label is not None and len(indices) > max_per_label:
            indices = rng.sample(indices, int(max_per_label))
        selected.extend(indices)
    return np.array(sorted(selected), dtype=int)


heatmap_indices = stratified_indices(predicates, HEATMAP_MAX_PER_PREDICATE)
heatmap_predicates = [predicates[i] for i in heatmap_indices]
heatmap_original = original_sim[np.ix_(heatmap_indices, heatmap_indices)]
heatmap_refined = refined_sim[np.ix_(heatmap_indices, heatmap_indices)]

vmin = min(float(heatmap_original.min()), float(heatmap_refined.min()))
vmax = max(float(heatmap_original.max()), float(heatmap_refined.max()))
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

for ax, matrix, title in [
    (axes[0], heatmap_original, "Original CLIP text"),
    (axes[1], heatmap_refined, f"CDP/SVD remove top {CDP_REMOVE_COMPONENTS}"),
]:
    off_mean = off_diagonal_values(matrix).mean()
    im = ax.imshow(matrix, cmap="coolwarm", vmin=vmin, vmax=vmax, interpolation="nearest", aspect="auto")
    ax.set_title(f"{title}\nmean off-diagonal cosine = {off_mean:.4f}")
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(im, ax=axes, shrink=0.85, label="cosine similarity")
fig.suptitle(f"Triplet Text Similarity Heatmaps ({len(heatmap_indices)} sampled triplets)", y=1.03)
heatmap_path = OUTPUT_DIR / "similarity_heatmaps.png"
fig.savefig(heatmap_path, bbox_inches="tight")
plt.show()
print("saved:", heatmap_path)

In [ ]:
original_values = off_diagonal_values(original_sim)
refined_values = off_diagonal_values(refined_sim)

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(min(original_values.min(), refined_values.min()), max(original_values.max(), refined_values.max()), 80)

if sns is not None:
    sns.histplot(original_values, bins=bins, stat="density", element="step", fill=False, linewidth=1.5, label="Original", ax=ax)
    sns.histplot(refined_values, bins=bins, stat="density", element="step", fill=False, linewidth=1.5, label="CDP/SVD", ax=ax)
    try:
        sns.kdeplot(original_values, color="C0", linewidth=1.2, ax=ax)
        sns.kdeplot(refined_values, color="C1", linewidth=1.2, ax=ax)
    except Exception:
        pass
else:
    ax.hist(original_values, bins=bins, density=True, histtype="step", linewidth=1.5, label="Original")
    ax.hist(refined_values, bins=bins, density=True, histtype="step", linewidth=1.5, label="CDP/SVD")

ax.axvline(original_values.mean(), color="C0", linestyle="--", linewidth=1.2)
ax.axvline(refined_values.mean(), color="C1", linestyle="--", linewidth=1.2)
ax.set_title("Off-Diagonal Cosine Similarity Distribution")
ax.set_xlabel("cosine similarity")
ax.set_ylabel("density")
ax.legend(frameon=True)
fig.tight_layout()
distribution_path = OUTPUT_DIR / "similarity_distribution.png"
fig.savefig(distribution_path, bbox_inches="tight")
plt.show()
print("saved:", distribution_path)

## PCA and Optional t-SNE

PCA is implemented with NumPy SVD so it does not require scikit-learn. t-SNE is optional and uses scikit-learn when available.

In [ ]:
def pca_2d(x):
    centered = x - x.mean(axis=0, keepdims=True)
    _, singular_values, vt = np.linalg.svd(centered, full_matrices=False)
    coords = centered @ vt[:2].T
    explained = singular_values ** 2 / np.maximum(np.sum(singular_values ** 2), 1e-12)
    return coords.astype(np.float32), explained[:2]


def make_plot_df(coords, method, feature_name):
    df = triplets_df.copy()
    df["x"] = coords[:, 0]
    df["y"] = coords[:, 1]
    df["method"] = method
    df["feature"] = feature_name
    return df


def save_scatter(df, title, path, max_legend_items=20):
    if px is not None:
        fig = px.scatter(
            df,
            x="x",
            y="y",
            color="predicate",
            hover_data=["subject", "predicate", "object", "text"],
            opacity=0.72,
            title=title,
            width=980,
            height=720,
        )
        fig.update_traces(marker={"size": 5, "line": {"width": 0}})
        fig.update_layout(legend_title_text="Predicate", legend={"itemsizing": "constant"})
        fig.show()

    predicates_order = df["predicate"].value_counts().index.tolist()
    color_map = {pred: idx for idx, pred in enumerate(predicates_order)}
    colors = df["predicate"].map(color_map).to_numpy()
    cmap = plt.get_cmap("tab20", max(len(predicates_order), 1))

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(df["x"], df["y"], c=colors, cmap=cmap, s=12, alpha=0.68, linewidths=0)
    ax.set_title(title)
    ax.set_xlabel("component 1")
    ax.set_ylabel("component 2")

    handles = []
    for pred in predicates_order[:max_legend_items]:
        handles.append(plt.Line2D([0], [0], marker="o", color="w", label=pred, markerfacecolor=cmap(color_map[pred]), markersize=6))
    if handles:
        ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True, title="Top predicates")
    ax.grid(True, linewidth=0.4, alpha=0.35)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print("saved:", path)


pca_coords, pca_explained = pca_2d(refined_embeddings)
pca_df = make_plot_df(pca_coords, "PCA", "CDP/SVD refined")
pca_title = f"PCA of Triplet Text Embeddings after CDP (explained {pca_explained[0]:.2%}, {pca_explained[1]:.2%})"
save_scatter(pca_df, pca_title, OUTPUT_DIR / "pca_scatter.png")

In [ ]:
if USE_TSNE:
    try:
        from sklearn.manifold import TSNE
        perplexity = min(TSNE_PERPLEXITY, max(5, (len(refined_embeddings) - 1) // 3))
        print("running t-SNE with perplexity:", perplexity)
        tsne = TSNE(
            n_components=2,
            perplexity=perplexity,
            init="pca",
            learning_rate="auto",
            random_state=RANDOM_SEED,
            metric="cosine",
        )
        tsne_coords = tsne.fit_transform(refined_embeddings).astype(np.float32)
        tsne_df = make_plot_df(tsne_coords, "t-SNE", "CDP/SVD refined")
        save_scatter(tsne_df, "t-SNE of Triplet Text Embeddings after CDP", OUTPUT_DIR / "tsne_scatter.png")
    except Exception as exc:
        print("Skipping t-SNE because sklearn TSNE is unavailable or failed:", repr(exc))
else:
    print("USE_TSNE is False; skipped t-SNE.")

## Outputs

The generated files are saved under `visualization/outputs/triplet_text_similarity/`.

In [ ]:
output_files = [
    OUTPUT_DIR / "similarity_heatmaps.png",
    OUTPUT_DIR / "similarity_distribution.png",
    OUTPUT_DIR / "pca_scatter.png",
    OUTPUT_DIR / "tsne_scatter.png",
    EMBEDDING_CACHE_PATH,
]

for path in output_files:
    print(("OK  " if path.exists() else "MISS"), path)

display(summary_df)
display(predicate_counts_df.describe(include="all"))